In [27]:
import pandas as pd

data = pd.read_csv('day18_regression.csv')

data.head()

,shipment_id,order_date,distance_km,weight_kg,traffic_level,weather,vehicle_type,priority,driver_experience_years,warehouse_wait_min,package_type,temperature_c,delivery_time_min
0,SHP00001,2026-05-24,23.9,5.0,high,clear,car,normal,5.5,22.5,fragile,1.2,94.4
1,SHP00002,2026-07-06,122.1,10.8,high,clear,car,express,0.0,12.6,large,28.4,166.0
2,SHP00003,2026-07-03,40.5,8.3,high,clear,van,normal,5.1,8.8,large,18.6,104.6
3,SHP00004,2026-06-08,100.2,7.2,medium,clear,van,express,8.7,22.8,large,1.7,144.2
4,SHP00005,2026-07-05,87.5,14.8,medium,clear,car,express,7.7,87.6,small,2.3,178.2


In [28]:
print(data.shape)
print(data.info())
print(data.isnull().sum())

(520, 13)
<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   shipment_id              520 non-null    str    
 1   order_date               520 non-null    str    
 2   distance_km              520 non-null    float64
 3   weight_kg                502 non-null    float64
 4   traffic_level            520 non-null    str    
 5   weather                  506 non-null    str    
 6   vehicle_type             520 non-null    str    
 7   priority                 520 non-null    str    
 8   driver_experience_years  500 non-null    float64
 9   warehouse_wait_min       520 non-null    float64
 10  package_type             508 non-null    str    
 11  temperature_c            520 non-null    float64
 12  delivery_time_min        520 non-null    float64
dtypes: float64(6), str(7)
memory usage: 52.9 KB
None
shipment_id                 0
or

In [29]:
data['order_date'] = pd.to_datetime(data['order_date'])

data['order_month'] = data['order_date'].dt.month
data['order_day'] = data['order_date'].dt.day
data['order_dayofweek'] = data['order_date'].dt.day_of_week

data = data.drop(columns=['order_date'])

In [30]:
data.head()

,shipment_id,distance_km,weight_kg,traffic_level,weather,vehicle_type,priority,driver_experience_years,warehouse_wait_min,package_type,temperature_c,delivery_time_min,order_month,order_day,order_dayofweek
0,SHP00001,23.9,5.0,high,clear,car,normal,5.5,22.5,fragile,1.2,94.4,5,24,6
1,SHP00002,122.1,10.8,high,clear,car,express,0.0,12.6,large,28.4,166.0,7,6,0
2,SHP00003,40.5,8.3,high,clear,van,normal,5.1,8.8,large,18.6,104.6,7,3,4
3,SHP00004,100.2,7.2,medium,clear,van,express,8.7,22.8,large,1.7,144.2,6,8,0
4,SHP00005,87.5,14.8,medium,clear,car,express,7.7,87.6,small,2.3,178.2,7,5,6


In [31]:
data['weight_kg'] = data['weight_kg'].fillna(data['weight_kg'].median())
data['driver_experience_years'] = data['driver_experience_years'].fillna(data['driver_experience_years'].median())
data['weather'] = data['weather'].fillna(data['weather'].mode()[0])
data['package_type'] = data['package_type'].fillna(data['package_type'].mode()[0])

In [32]:
data.head()

,shipment_id,distance_km,weight_kg,traffic_level,weather,vehicle_type,priority,driver_experience_years,warehouse_wait_min,package_type,temperature_c,delivery_time_min,order_month,order_day,order_dayofweek
0,SHP00001,23.9,5.0,high,clear,car,normal,5.5,22.5,fragile,1.2,94.4,5,24,6
1,SHP00002,122.1,10.8,high,clear,car,express,0.0,12.6,large,28.4,166.0,7,6,0
2,SHP00003,40.5,8.3,high,clear,van,normal,5.1,8.8,large,18.6,104.6,7,3,4
3,SHP00004,100.2,7.2,medium,clear,van,express,8.7,22.8,large,1.7,144.2,6,8,0
4,SHP00005,87.5,14.8,medium,clear,car,express,7.7,87.6,small,2.3,178.2,7,5,6


In [33]:
data.shape

(520, 15)

In [34]:
Q1 = data['warehouse_wait_min'].quantile(0.25)
Q3 = data['warehouse_wait_min'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

data = data[
  (data['warehouse_wait_min']>=lower)&
  (data['warehouse_wait_min']<=upper)
  ]

data.shape


(512, 15)

In [35]:
data['traffic_level'] = data['traffic_level'].map({
  'low':0,
  'medium':1,
  'high':2
})

data = pd.get_dummies(data, columns=['weather','vehicle_type','priority','package_type'], dtype=int)

In [36]:
data.head()

,shipment_id,distance_km,weight_kg,traffic_level,driver_experience_years,warehouse_wait_min,temperature_c,delivery_time_min,order_month,order_day,...,vehicle_type_bike,vehicle_type_car,vehicle_type_van,priority_express,priority_normal,priority_same_day,package_type_fragile,package_type_large,package_type_medium,package_type_small
0,SHP00001,23.9,5.0,2,5.5,22.5,1.2,94.4,5,24,...,0,1,0,0,1,0,1,0,0,0
1,SHP00002,122.1,10.8,2,0.0,12.6,28.4,166.0,7,6,...,0,1,0,1,0,0,0,1,0,0
2,SHP00003,40.5,8.3,2,5.1,8.8,18.6,104.6,7,3,...,0,0,1,0,1,0,0,1,0,0
3,SHP00004,100.2,7.2,1,8.7,22.8,1.7,144.2,6,8,...,0,0,1,1,0,0,0,1,0,0
5,SHP00006,89.1,8.5,1,6.2,8.2,-2.8,155.8,5,20,...,0,0,1,0,1,0,0,0,1,0


In [39]:
x = data.drop(columns=['shipment_id', 'delivery_time_min'])

In [40]:
y = data['delivery_time_min']

print(x.shape)
print(y.shape)

(512, 22)
(512,)


In [42]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
  x,
  y,
  test_size=0.2,
  random_state=42 # 회귀문제 straitify 사용안함
)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)


(409, 22)
(103, 22)
(409,)
(103,)


In [43]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print(x_train_scaled.shape)
print(x_test_scaled.shape)

(409, 22)
(103, 22)


In [45]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(random_state=42)

lr.fit(x_train_scaled, y_train)
dt.fit(x_train, y_train)
rf.fit(x_train, y_train);



In [46]:
lr_pred = lr.predict(x_test_scaled)
dt_pred = dt.predict(x_test)
rf_pred = rf.predict(x_test)

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_regression(y_test, pred):
  mae = mean_absolute_error(y_test, pred)
  mse = mean_squared_error(y_test, pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, pred)

  print("MAE :", mae)
  print("MSE :", mse)
  print("RMSE:", rmse)
  print("R2  :", r2)
  print()


evaluate_regression(y_test, lr_pred)
evaluate_regression(y_test, dt_pred)
evaluate_regression(y_test, rf_pred)

MAE : 7.348477115284269
MSE : 95.24658030617674
RMSE: 9.759435450177266
R2  : 0.9646469677139141

MAE : 18.67087378640776
MSE : 580.6369902912621
RMSE: 24.096410319615288
R2  : 0.7844827793472859

MAE : 13.563902912621359
MSE : 293.76459071844636
RMSE: 17.139562150721538
R2  : 0.8909622893883095



In [58]:
# 딥러닝 회귀
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input


model = Sequential([
  Input(shape=(x_train_scaled.shape[1],)),
  Dense(64, activation='relu'),
  Dropout(0.2),
  Dense(32, activation='relu'),
  Dense(1, activation='linear')
])

model.compile(
  optimizer='adam',
  loss='mean_squared_error',
  metrics=['mae']
)


In [59]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

estop = EarlyStopping(
  monitor='val_loss',
  patience=10,
  restore_best_weights=True
)

checkpoint = ModelCheckpoint(
  filepath='best_regression_model.keras',
  monitor='val_loss',
  save_best_only=True
)

history = model.fit(
  x_train_scaled,
  y_train,
  epochs=100,
  batch_size=32,
  validation_split=0.2,
  callbacks=[estop, checkpoint]
)



Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 20395.3750 - mae: 134.5874 - val_loss: 19849.8594 - val_mae: 134.0329
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 20217.3867 - mae: 133.9060 - val_loss: 19676.8438 - val_mae: 133.3821
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 20034.3203 - mae: 133.2382 - val_loss: 19491.9961 - val_mae: 132.6851
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 19826.9121 - mae: 132.4474 - val_loss: 19264.9082 - val_mae: 131.8258
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 19560.6523 - mae: 131.4777 - val_loss: 18977.5645 - val_mae: 130.7385
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 19218.9180 - mae: 130.1937 - val_loss: 18619.3379 - val_mae: 129.3801
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 18815.3359 - mae: 128.7050 - val_loss: 18184.7891 - val_mae: 127.7227
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 18321.9668 - mae: 126.8007 - val_los

In [60]:
# 회귀 딥러닝 mae, mse, rmse, r2
dl_pred = model.predict(x_test_scaled)

mae = mean_absolute_error(y_test, dl_pred)
mse = mean_squared_error(y_test, dl_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, dl_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)
print()

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
MAE : 11.944997687478669
MSE : 224.1950101242366
RMSE: 14.973142960789382
R2  : 0.9167846928905699

